# Vision-model re-ranking (Kaggle)

Re-ranks BLIP's top 30 for each round-0 caption with `qwen2.5vl:7b` (yes/no probability prompt).
See the docstring of `vision_rerank_experiment.py` for the design and `REWRITE_FUSION_VISION_RERANK.md` for results.

**Before you start** (right-hand panel, *Settings*):
- **Accelerator:** *GPU T4 x2*.
- **Internet:** *On*, needed for Ollama and the COCO photos.

Both need a phone-verified Kaggle account.

**Files:** add a dataset with these files from the `vision_model_reranker` branch:
- `vision_rerank_experiment.py`
- a shortlist file: `runs/vision_rerank/shortlists_top30.json` (pilot, 47 targets) or
  `shortlists_top30_all.json` (all 176 reachable targets; build it with `python vision_rerank_experiment.py shortlist --all`)
- optional: saved answers `data/vlm_scores/qwen2.5vl_7b_yes_prob_*.json`, so they are not asked again

To add it: right-hand panel → *Add Input* → *Upload* (or *+ New Dataset*), drop the files in, keep it private, then create it.
Kaggle resets idle sessions and deletes `/kaggle/working`: rerun the cells from the top after a reset.

## 1. Copy the files into the working folder

In [ ]:
import glob, os, shutil
os.chdir("/kaggle/working")
os.makedirs("data/vlm_scores", exist_ok=True)
for name in ["vision_rerank_experiment.py", "shortlists_top30*.json", "qwen2.5vl_7b_yes_prob_*.json"]:
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not found and name.endswith(".py"):
        raise FileNotFoundError(f"{name} not found under /kaggle/input: add it as a dataset (see above).")
    for path in found:
        shutil.copy(path, "data/vlm_scores/" if "yes_prob" in name else os.path.basename(path))
        print("copied", path)
assert "split_rerank" in open("vision_rerank_experiment.py").read(), "old script version: update the dataset"

## 2. Install Ollama and start it
The `yes_prob` prompt reads token log-probabilities, which need Ollama 0.12.11 or newer, so this installs the latest.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > install.log 2>&1; tail -2 install.log
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess, time, urllib.request
import os
# One GPU and the CUDA 12 build: with Kaggle's two T4s and the default CUDA 13 build,
# Ollama hung while loading the model (2026-11-14, versions 0.12.3 and latest).
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "OLLAMA_LLM_LIBRARY": "cuda_v12"}
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://localhost:11434/api/version").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see ollama.log")

## 3. Download the vision model

In [ ]:
!ollama pull qwen2.5vl:7b > /dev/null 2>&1
!ollama list

## 4. Optional: compare prompts on a few targets (a few minutes)

Scores the shortlists of 6 targets (2 from each rank group) with each prompt and prints the target's
score against the other photos. Already done: `yes_prob` with the 7B model separated the target best.
Afterwards, `ollama ps` should say `100% GPU`.

In [ ]:
!python vision_rerank_experiment.py compare shortlists_top30.json --targets 6 --prompts yes_prob
!ollama ps

## 5. Run the re-ranking

Pilot (47 targets × 30 photos): about 80 min on a T4. All reachable targets (`shortlists_top30_all.json`, 176 × 30):
about 4 h more, reusing the pilot's answers. Answers are saved as they come in (`data/vlm_scores/`), so
if the cell stops, run it again and it continues. The first line says how many answers are reused.

In [ ]:
!python vision_rerank_experiment.py rerank shortlists_top30.json

## 6. Download the results

Everything in `/kaggle/working` shows up in the right-hand panel under *Output*. Download
`vision_rerank_results.zip` from there, and unzip it in the repo root on your laptop. That
restores `runs/vision_rerank/<timestamp>/` and the saved answers in `data/vlm_scores/`.

Download before stopping the session: an interactive session's files are deleted when it ends.

In [ ]:
!zip -qr vision_rerank_results.zip runs/vision_rerank data/vlm_scores
!ls -lh vision_rerank_results.zip